In [1]:
import pandas as pd

customers = pd.read_csv("customers.csv")
transactions = pd.read_csv("transactions.csv")

In [3]:
customers.shape
transactions.shape

customers.info()
transactions.info()

customers.isna().sum()
transactions.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     50000 non-null  int64  
 1   country         50000 non-null  str    
 2   first_purchase  50000 non-null  str    
 3   last_purchase   50000 non-null  str    
 4   n_orders        50000 non-null  float64
 5   total_spent     50000 non-null  float64
 6   avg_basket      50000 non-null  float64
 7   recency_days    50000 non-null  float64
 8   tenure_days     50000 non-null  float64
dtypes: float64(5), int64(1), str(3)
memory usage: 3.4 MB
<class 'pandas.DataFrame'>
RangeIndex: 1837137 entries, 0 to 1837136
Data columns (total 8 columns):
 #   Column        Dtype  
---  ------        -----  
 0   invoice_id    str    
 1   customer_id   float64
 2   product_code  str    
 3   product_name  str    
 4   quantity      float64
 5   unit_price    float64
 6   invoice_date  str    
 7   c

invoice_id           0
customer_id     418258
product_code         0
product_name      7542
quantity         16187
unit_price           0
invoice_date         0
country              0
dtype: int64

In [4]:
# Nombre total de clients dans customers.csv
n_customers_crm = customers['customer_id'].nunique()
n_customers_crm

50000

In [5]:
# Nombre de clients distincts dans transactions (en excluant les NaN)
n_customers_transactions = transactions['customer_id'].dropna().nunique()
n_customers_transactions

49146

In [6]:
# Clients du CRM absents des transactions
crm_ids = set(customers['customer_id'])
transaction_ids = set(transactions['customer_id'].dropna())

clients_without_transactions = crm_ids - transaction_ids
len(clients_without_transactions)

1950

In [7]:
# Vérifier les IDs présents dans transactions mais absents du CRM
extra_ids = transaction_ids - crm_ids
len(extra_ids)

1096

In [ ]:
# Nombre de transactions
n_lines = transactions.shape[0]
n_lines

1837137

In [9]:
# nombre de factures uniques
n_invoices = transactions['invoice_id'].nunique()
n_invoices

253545

In [10]:
# Nombre de transactions anonymes
n_anonymous_transactions = transactions['customer_id'].isna().sum()
n_anonymous_transactions

np.int64(418258)

In [ ]:
# Proportion de transactiions anonymes
anonymous_ratio = n_anonymous_transactions / n_lines
anonymous_ratio

np.float64(0.22766837748083021)

In [13]:
# Conversion des dates
customers['first_purchase'] = pd.to_datetime(customers['first_purchase'])
customers['last_purchase'] = pd.to_datetime(customers['last_purchase'])
transactions['invoice_date'] = pd.to_datetime(transactions['invoice_date'])

In [ ]:
# Période des transactions
transactions['invoice_date'].min(), transactions['invoice_date'].max()

(Timestamp('2007-07-06 12:20:00'), Timestamp('2011-12-09 12:50:00'))

In [16]:
# Période des customers
customers['first_purchase'].min(), customers['last_purchase'].max()

(Timestamp('2007-05-18 12:20:00'), Timestamp('2011-12-09 12:20:00'))

In [17]:
transactions['invoice_year'] = transactions['invoice_date'].dt.year
transactions['invoice_year'].value_counts().sort_index()

invoice_year
2007       302
2008     12823
2009    141250
2010    857572
2011    825190
Name: count, dtype: int64

In [ ]:
# Valeurs manquantes dans customers
customers_missing = pd.DataFrame({
    "missing_count": customers.isna().sum(),
    "missing_ratio": customers.isna().mean()
})

customers_missing

,missing_count,missing_ratio
customer_id,0,0.0
country,0,0.0
first_purchase,0,0.0
last_purchase,0,0.0
n_orders,0,0.0
total_spent,0,0.0
avg_basket,0,0.0
recency_days,0,0.0
tenure_days,0,0.0


In [19]:
# Valeurs manquantes dans transaction
transactions_missing = pd.DataFrame({
    "missing_count": transactions.isna().sum(),
    "missing_ratio": transactions.isna().mean()
})

transactions_missing

,missing_count,missing_ratio
invoice_id,0,0.000000
customer_id,418258,0.227668
product_code,0,0.000000
product_name,7542,0.004105
quantity,16187,0.008811
unit_price,0,0.000000
invoice_date,0,0.000000
country,0,0.000000
invoice_year,0,0.000000


In [21]:
# Type transactions
transactions.dtypes

invoice_id                 str
customer_id            float64
product_code               str
product_name               str
quantity               float64
unit_price             float64
invoice_date    datetime64[us]
country                    str
invoice_year             int32
dtype: object

In [23]:
# type customers
customers.dtypes

customer_id                int64
country                      str
first_purchase    datetime64[us]
last_purchase     datetime64[us]
n_orders                 float64
total_spent              float64
avg_basket               float64
recency_days             float64
tenure_days              float64
dtype: object

In [24]:
transactions['customer_id'].describe()

count    1.418879e+06
mean     2.241318e+04
std      1.254708e+04
min      1.234600e+04
25%      1.489500e+04
50%      1.749100e+04
75%      2.179900e+04
max      6.344100e+04
Name: customer_id, dtype: float64

In [25]:
# Doublons customers
customers.duplicated().sum()

np.int64(0)

In [26]:
# Doublon transaction
transactions.duplicated().sum()

np.int64(34522)

In [27]:
transactions.duplicated(subset=[
    'invoice_id', 
    'product_code', 
    'quantity', 
    'unit_price'
]).sum()

np.int64(34701)

In [28]:
incoherent_dates = customers[
    customers['first_purchase'] > customers['last_purchase']
]

len(incoherent_dates)

0